###Sử dụng google colab nếu máy tính cá nhân bạn không đủ 16GB RAM GPU
###Use google colab if your laptop or pc don't have of 16GB RAM GPU


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
!pip -q uninstall -y transformers
!pip -q install transformers==5.0.0 accelerate peft bitsandbytes datasets



   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.1/10.1 MB 72.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 15.8 MB/s eta 0:00:00


In [ ]:
import transformers, inspect
from transformers import TrainingArguments

print("transformers version =", transformers.__version__)
print("TrainingArguments module =", TrainingArguments.__module__)
print("TrainingArguments init signature =")
print(inspect.signature(TrainingArguments.__init__))


transformers version = 5.0.0
TrainingArguments module = transformers.training_args
TrainingArguments init signature =
(self, output_dir: str | None = None, do_train: bool = False, do_eval: bool = False, do_predict: bool = False, eval_strategy: transformers.trainer_utils.IntervalStrategy | str = 'no', prediction_loss_only: bool = False, per_device_train_batch_size: int = 8, per_device_eval_batch_size: int = 8, gradient_accumulation_steps: int = 1, eval_accumulation_steps: int | None = None, eval_delay: float = 0, torch_empty_cache_steps: int | None = None, learning_rate: float = 5e-05, weight_decay: float = 0.0, adam_beta1: float = 0.9, adam_beta2: float = 0.999, adam_epsilon: float = 1e-08, max_grad_norm: float = 1.0, num_train_epochs: float = 3.0, max_steps: int = -1, lr_scheduler_type: transformers.trainer_utils.SchedulerType | str = 'linear', lr_scheduler_kwargs: dict | str | None = None, warmup_ratio: float | None = None, warmup_steps: float = 0, log_level: str = 'passive', log_lev

#set up

In [ ]:

from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
import torch

model_name = "Qwen/Qwen2.5-Math-7B"

#4-bit quantization (QLoRA)
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",        #NormalFloat4 (NF4) to increase accuracy
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.bfloat16  #bfloat16 to save VRAM
)

#tokenizer
tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)
tokenizer.padding_side = "right"
if tokenizer.pad_token_id is None:
    tokenizer.pad_token_id = tokenizer.eos_token_id

#model 4-bit
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,
    device_map="auto",
    torch_dtype=torch.bfloat16,
    trust_remote_code=True
)

model.config.use_cache = False
model.gradient_checkpointing_enable()



/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/678 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

...

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/138 [00:00<?, ?B/s]

In [ ]:
import pandas as pd

df = pd.read_csv("/content/drive/MyDrive/Khoa_luan_2025_2026/Data_source/DataAugmentation.csv")
#prompt for finetune
df['prompt'] = df['Instruction'].astype(str) + " " + df['Latex Input'].astype(str)

df['output'] = df['Response'].astype(str)

df = df.dropna(subset=["prompt", "output"])
print("Total samples:", len(df))
print("Sample prompt:\n", df.iloc[0]["prompt"])
print("Sample response:\n", df.iloc[0]["output"])



Total samples: 5360
Sample prompt:
 Hãy giải giúp tôi bài sau: Cho hai biến cố $A$ và $B$. Biết rằng $P(A\cup B)=0.88$; $P(A)=0.6$; $P(B)=0.7$.Tính $P(A\cap B)$ và chứng tỏ $A$ và $B$ độc lập với nhau.
Sample response:
 /buoc1 Sử dụng công thức cộng xác suất: $P(AB) = P(A) + P(B) - P(A + B)$.
/buoc2 Thay số: $P(AB) = 0.6 + 0.7 - 0.88 = 0.42$.
/buoc3 Xét tích xác suất: $P(A) \cdot P(B) = 0.6 \cdot 0.7 = 0.42$.
/ketluan Vì $P(AB) = P(A) \cdot P(B) = 0.42$ nên $A$ và $B$ độc lập với nhau.
/dapan độc lập


In [ ]:
df[['prompt','output']]

,prompt,output
0,Hãy giải giúp tôi bài sau: Cho hai biến cố $A$...,/buoc1 Sử dụng công thức cộng xác suất: $P(AB)...
1,Hãy giải giúp tôi bài sau: Cho hai biến cố $A$...,/buoc1 Sử dụng công thức cộng xác suất: $P(A \...
2,Hãy giải giúp tôi bài sau: Cho hai biến cố $A$...,/buoc1 Sử dụng công thức cộng xác suất: $P(A \...
3,Hãy giải giúp tôi bài sau: Cho hai biến cố $A$...,/buoc1 Sử dụng công thức cộng xác suất: $P(A \...
4,Hãy giải giúp tôi bài sau: Cho hai biến cố $A$...,/buoc1 Sử dụng công thức cộng xác suất: $P(A \...
...,...,...
5355,Hãy giải giúp tôi bài sau: Cho biến ngẫu nhiên...,/buoc1: Lập phân phối lề của $Y$.\n$$P(Y=-3)=0...
5356,Hãy giải giúp tôi bài sau: Cho biến ngẫu nhiên...,/buoc1: Lập phân phối lề của $Y$.\n$$P(Y=-2)=0...
5357,Hãy giải giúp tôi bài sau: Cho biến ngẫu nhiên...,/buoc1: Lập phân phối lề của $Y$.\n$$P(Y=-3)=0...
5358,Hãy giải giúp tôi bài sau: Cho biến ngẫu nhiên...,/buoc1: Lập phân phối lề của $Y$.\n$$P(Y=-4)=0...


In [ ]:
from datasets import Dataset, DatasetDict
from sklearn.model_selection import train_test_split


# train90%, test 10%
train_df, val_df = train_test_split(df, test_size=0.1, random_state=42)
train_data = Dataset.from_pandas(train_df[["prompt", "output"]])
val_data   = Dataset.from_pandas(val_df[["prompt", "output"]])

#tokenize prompt+output
def preprocess(example):
    prompt_text = example['prompt']
    output_text = example['output']
    full_text = prompt_text + " " + output_text

    tokens = tokenizer(full_text, truncation=True, max_length=512)
    input_ids = tokens['input_ids']
    #Length limit
    prompt_ids = tokenizer(prompt_text, truncation=True, max_length=512)['input_ids']
    prompt_length = len(prompt_ids)
    labels = input_ids.copy()
    labels[:prompt_length] = [-100] * prompt_length

    return {"input_ids": input_ids, "labels": labels}

#preprocessing
train_enc = train_data.map(preprocess, remove_columns=train_data.column_names)
val_enc = val_data.map(preprocess, remove_columns=val_data.column_names)

print("Số mẫu train:", len(train_enc))
print("Số mẫu test:", len(val_enc))
print("Ví dụ tokenized mẫu đầu tiên (input_ids):", train_enc[0]['input_ids'][:20])
print("Ví dụ tokenized mẫu đầu tiên (labels):", train_enc[0]['labels'][:20])


Map:   0%|          | 0/4824 [00:00<?, ? examples/s]

Map:   0%|          | 0/536 [00:00<?, ? examples/s]

Số mẫu train: 4824
Số mẫu test: 536
Ví dụ tokenized mẫu đầu tiên (input_ids): [39, 3202, 88, 128486, 128430, 128296, 128601, 32154, 25, 32580, 400, 56, 1124, 14781, 451, 7, 16, 26, 16, 8]
Ví dụ tokenized mẫu đầu tiên (labels): [-100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100]


In [ ]:
import torch
from torch.nn.utils.rnn import pad_sequence

#Collator: pad input_ids bằng pad_token_id, pad labels bằng -100
def data_collator(batch):
    input_id_list = [torch.tensor(ex["input_ids"], dtype=torch.long) for ex in batch]
    labels_list = [torch.tensor(ex["labels"], dtype=torch.long) for ex in batch]
    #Pad sequences
    input_ids_padded = pad_sequence(input_id_list, batch_first=True, padding_value=tokenizer.pad_token_id)
    labels_padded = pad_sequence(labels_list, batch_first=True, padding_value=-100)
    #attention mask (1 for token, 0 for pad)
    attention_mask = (input_ids_padded != tokenizer.pad_token_id).long()
    return {"input_ids": input_ids_padded, "labels": labels_padded, "attention_mask": attention_mask}
#test
batch_test = [train_enc[i] for i in range(2)]
collated = data_collator(batch_test)
print("Kích thước batch input_ids:", collated['input_ids'].shape)
print("Ví dụ batch input_ids dòng 0:", collated['input_ids'][0][:10])
print("Ví dụ batch labels dòng 0:", collated['labels'][0][:10])


Kích thước batch input_ids: torch.Size([2, 198])
Ví dụ batch input_ids dòng 0: tensor([    39,   3202,     88, 128486, 128430, 128296, 128601,  32154,     25,
         32580])
Ví dụ batch labels dòng 0: tensor([-100, -100, -100, -100, -100, -100, -100, -100, -100, -100])


In [ ]:
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

#model 4 bit
model = prepare_model_for_kbit_training(model)

#configuration QLoRA
lora_config = LoraConfig(
    r=16,              # rank adapter
    lora_alpha=32,     #scale qlora
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

#adapter QLoRA
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()


trainable params: 40,370,176 || all params: 7,655,986,688 || trainable%: 0.5273


In [ ]:
import math
from transformers import TrainingArguments, Trainer

model.train()

training_args = TrainingArguments(
    output_dir="/content/drive/MyDrive/Khoa_luan_2025_2026/qwen3d",
    num_train_epochs=0.5,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,  #2 batch tích lũy gradient 4 bước
    learning_rate=2e-4,
    weight_decay=0.001,
    optim="paged_adamw_32bit",
    lr_scheduler_type="cosine",
    warmup_ratio=0.03,
    logging_steps=10,
    eval_strategy="epoch",
    save_strategy="steps",
    save_steps=10,
    fp16=True  #use float16 for gradient
)


warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


In [ ]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_enc,
    eval_dataset=val_enc,
    data_collator=data_collator
)

In [ ]:
#trainer.train()


In [ ]:
trainer.train(resume_from_checkpoint="/content/drive/MyDrive/Khoa_luan_2025_2026/qwen3d/checkpoint-550")

/usr/local/lib/python3.12/dist-packages/torch/_dynamo/eval_frame.py:1181: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)


Epoch,Training Loss,Validation Loss


OutOfMemoryError: CUDA out of memory. Tried to allocate 2.32 GiB. GPU 0 has a total capacity of 14.56 GiB of which 1.77 GiB is free. Including non-PyTorch memory, this process has 12.79 GiB memory in use. Of the allocated memory 8.99 GiB is allocated by PyTorch, and 3.67 GiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)

## save full model

In [ ]:
import os
import json
import torch
import transformers
import peft

adapter_dir = "/content/drive/MyDrive/Khoa_luan_2025_2026/qwen2_5_math_adapter"

trainer.model.eval()

# save adapter + adapter_config
trainer.model.save_pretrained(
    adapter_dir,
    safe_serialization=True)

tokenizer.save_pretrained(adapter_dir)

#meta data to load the same as finetune configuration
runtime_meta = {
    "base_model_name": "Qwen/Qwen2.5-Math-7B",
    "quantization": {
        "load_in_4bit": True,
        "bnb_4bit_quant_type": "nf4",
        "bnb_4bit_use_double_quant": True,
        "bnb_4bit_compute_dtype": "bfloat16"
    },
    "library_versions": {
        "transformers": transformers.__version__,
        "peft": peft.__version__,
        "torch": torch.__version__
    }
}

with open(os.path.join(adapter_dir, "runtime_meta.json"), "w", encoding="utf-8") as f:
    json.dump(runtime_meta, f, ensure_ascii=False, indent=2)

print(f"Đã lưu adapter tại: {adapter_dir}")


Đã lưu adapter chuẩn tại: /content/drive/MyDrive/Khoa_luan_2025_2026/qwen2_5_math_adapter


###Test


In [ ]:
from transformers import StoppingCriteria, StoppingCriteriaList

class StopOnString(StoppingCriteria):
    def __init__(self, tokenizer, stop_string):
        self.stop_ids = tokenizer(stop_string, return_tensors="pt").input_ids[0]

    def __call__(self, input_ids, scores, **kwargs):
        if len(input_ids[0]) < len(self.stop_ids):
            return False
        return (input_ids[0][-len(self.stop_ids):] == self.stop_ids.to(input_ids.device)).all()

stopping_criteria = StoppingCriteriaList([
    StopOnString(tokenizer, "/dapan")
])

In [ ]:
generated_texts = []
test=[]
i=input("Nhập câu hỏi:")
i = "Hãy giúp tôi giải bài sau: " + "\n" + i + "\n"
test.append(i)
for sample in test:
  prompt = sample
  #gọi model
  inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
  outputs = model.generate(
      **inputs,top_p=0.9,
      max_new_tokens=512,
      temperature=0.0,
      stopping_criteria=stopping_criteria,
      do_sample=False, # do_sample=False
      eos_token_id=tokenizer.eos_token_id,
      pad_token_id=tokenizer.pad_token_id,
      early_stopping=True
    )

  output = tokenizer.decode(outputs[0], skip_special_tokens=True)
  output = output[len(prompt):].strip()
  print(output)
  if "/dapan" in output:
    output = output.split("/dapan")[0]
  generated_texts.append(output)
  print(f"Prompt: {prompt}\nGenerated answer: {output}")


Nhập câu hỏi:Cho hai biến cố $A$ và $B$. Biết rằng $P(A \cup B)=0.8; P(A)=0.6; P(B)=0.5$. Tính $P(A \cap B)$ và chứng tỏ $A$ và $B$ độc lập với nhau.
/buoc1 Sử dụng công thức cộng xác suất: $P(A \cap B) = P(A) + P(B) - P(A \cup B)$. /buoc2 Thay số: $P(A \cap B) = 0.6 + 0.5 - 0.8 = 0.3$. /buoc3 Xét tích xác suất: $P(A) \cdot P(B) = 0.6 \cdot 0.5 = 0.3$. /ketluan Vì $P(A \cap B) = P(A) \cdot P(B) = 0.3$ nên $A$ và $B$ độc lập với nhau. /dapan độc lập với nhau. /dapan 0.3. /dapan 0.3. /dapan độc lập với nhau. /dapan 0.3. /dapan 0.3. /dapan độc lập với nhau. /dapan 0.3. /dapan 0.3. /dapan độc lập với nhau. /dapan 0.3. /dapan 0.3. /dapan độc lập với nhau. /dapan 0.3. /dapan 0.3. /dapan độc lập với nhau. /dapan 0.3. /dapan 0.3. /dapan độc lập với nhau. /dapan 0.3. /dapan 0.3. /dapan độc lập với nhau. /dapan 0.3. /dapan 0.3. /dapan độc lập với nhau. /dapan 0.3. /dapan 0.3. /dapan độc lập với nhau. /dapan 0.3. /dapan 0.3. /dapan độc lập với nhau. /dapan 0.3. /dapan 0.3. /dapan độc lập với nhau